# Gold layer

The business view: one wide row per hour, joining weather, consumption, wind
production and price into a single table an analyst can query without knowing
anything about the sources.

Silver made the four sources describe the same world in the same way. Gold puts
them side by side and decides what the question actually is.

## Design decisions

**Weather is pivoted, not averaged.** Silver keeps four rows per hour, one per
observation point. A join needs one row per hour, so the four points become
eight columns. Averaging them away would have destroyed the regional signal:
Vaasa sits on the west coast where most of Finland's wind capacity is, and its
weather is the one most likely to explain wind production. A national mean cannot
answer that question.

**National means are carried as well.** Some questions are about the country
rather than a region. Temperature driving heating demand is national in
character; wind driving turbine output is not. Both shapes are present so
neither question requires a second table.

**The join is an inner join.** The four sources do not cover identical windows,
and with incremental loading they never will: weather stops six days back because
ERA5 publishes with that lag, while Fingrid and the price run to within hours of
now. An inner join trims the result to the period all four cover, without any
hand-written date bounds that would need maintaining as the window moves. An
outer join would have produced rows where half the columns are empty, which is
worse than not having the row.

The consequence is worth stating plainly: **this table always ends about six days
behind today**, and it is the weather source that decides that.

## Coverage

`2025-09-23 15:00` to `2026-09-17 23:00` UTC, 8,624 rows.

The start is set by Fingrid, the end by weather. This is the window in which every
statement made from this table is supported by all four sources.

## One hour is missing, and that is correct

The window spans 8,625 hourly points but the table has 8,624.

`2026-06-04 12:00` UTC is absent. Fingrid's wind production series is missing the
12:15 reading for that hour, so silver dropped the hour as incomplete rather than
averaging three quarters and presenting the result as equal to every other hour.
The inner join then removed that hour from all four sources.

The gap was found by tracing a single row count discrepancy through the pipeline:
`bronze_wind` held one reading fewer than `bronze_consumption`, `silver_wind` one
hour fewer, and gold one row fewer than the window implies. A `LAG` window
function over `silver_wind` located the exact hour.

This is the pipeline behaving as intended. The alternative would have been a
silent error: one hour whose average is computed differently from the other
8,623, with nothing to indicate it.

## Columns

| Column | Meaning |
| --- | --- |
| `time_utc` | Hour start, UTC. The join key and the source of truth. |
| `time_local` | Same instant in Europe/Helsinki, for reporting. |
| `FI_S_temp_c`, `FI_S_wind_ms` | Helsinki, south |
| `FI_W_temp_c`, `FI_W_wind_ms` | Vaasa, west coast, near most wind capacity |
| `FI_E_temp_c`, `FI_E_wind_ms` | Kuopio, east |
| `FI_N_temp_c`, `FI_N_wind_ms` | Oulu, north |
| `temp_avg_c`, `wind_avg_ms` | Unweighted mean across the four points |
| `consumption_mw` | Hourly mean electricity consumption, Finland |
| `wind_mw` | Hourly mean wind power generation, Finland |
| `price_eur_mwh` | Hourly mean day-ahead price, Finnish bidding zone |

## What this table does not establish

Any relationship visible here is **correlation, not causation**. Weather,
consumption and price move together for reasons this dataset cannot separate:
time of day, day of week, season, industrial activity, interconnector flows and
the behaviour of other Nordic bidding zones all act at once and none of them are
present as columns.

The four weather points are a deliberate simplification. Finland has 19 regions;
this uses four cities chosen to span the north/south temperature range and to
put one observation near the west coast wind capacity. It is not an
administrative or population-weighted division and should not be described as one.


In [0]:
from pyspark.sql import functions as F

SCHEMA = "workspace.energy_weather"

# Narrow each source to its join key and its measures
consumption = spark.table(f"{SCHEMA}.silver_consumption").select(
    "time_utc", "consumption_mw"
)
wind = spark.table(f"{SCHEMA}.silver_wind").select("time_utc", "wind_mw")
price = spark.table(f"{SCHEMA}.silver_price").select("time_utc", "price_eur_mwh")

weather = spark.table(f"{SCHEMA}.silver_weather")

# One row per hour, one column pair per observation point.
# The explicit value list keeps column order stable and lets Spark
# skip a scan it would otherwise need to discover the values.
weather_wide = (
    weather.groupBy("time_utc", "time_local")
    .pivot("area_id", ["FI_S", "FI_W", "FI_E", "FI_N"])
    .agg(
        F.first("temperature_c").alias("temp_c"),
        F.first("wind_speed_ms").alias("wind_ms"),
    )
)

# National means, for questions that are about the country rather than a region
weather_avg = weather.groupBy("time_utc").agg(
    F.avg("temperature_c").alias("temp_avg_c"),
    F.avg("wind_speed_ms").alias("wind_avg_ms"),
)

# Inner joins trim the result to the window all four sources cover
gold_hourly = (
    weather_wide.join(weather_avg, on="time_utc", how="inner")
    .join(consumption, on="time_utc", how="inner")
    .join(wind, on="time_utc", how="inner")
    .join(price, on="time_utc", how="inner")
)

print("Rows:", gold_hourly.count())
gold_hourly.select(
    F.min("time_utc").alias("first"), F.max("time_utc").alias("last")
).show(truncate=False)
gold_hourly.printSchema()

In [0]:
gold_hourly.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.energy_weather.gold_hourly"
)

print("Written:", spark.table("workspace.energy_weather.gold_hourly").count())

# Dimensional model

The same hours modelled as a star schema, alongside the wide `gold_hourly`
table above. Both are gold and they serve different readers. The wide table
feeds a dashboard with no joins. The star schema answers questions that need
stated grain, conformed dimensions and history.

## Two facts, because the grain differs

`fact_power_hour` is one row per hour. Consumption, wind production and price
are national figures.

`fact_weather_hour` is one row per hour **per area**. Temperature and wind speed
are local.

These are not merged into a single fact. Doing so would repeat every national
figure four times, and any `SUM` over it would be four times too large. One
fact, one grain.

When a question needs both, the finer fact is aggregated up to the coarser
grain first, as the report queries below do with a CTE. Joining two facts of
different grain directly is the classic modelling error known as a fan trap.

The one case where a direct join is safe is when the result is grouped by the
dimension that caused the fan-out, because the grouping undoes it. The
correlation query below does exactly that.

## Three dimensions

`dim_date`, one row per calendar day, with year, month, weekday, weekend flag
and season. The date is derived from **local** time, not UTC: an analyst asking
about Monday means the Finnish Monday. The facts derive `date_key` the same way,
otherwise the join would be off by the UTC offset.

`dim_area`, SCD1. City, region and a note on wind capacity. Nothing here changes
historically. If a city name were wrong, the correction is a fix, not an event,
so the old value should disappear.

`dim_wind_capacity`, SCD2. Installed onshore wind capacity for Finland, which
genuinely changes over time: 8,224 MW in 2025 and 9,330 MW in 2026. Overwriting
the old figure would silently recompute every 2025 capacity factor against a
capacity that did not exist yet.

## Why the capacity dimension needs a surrogate key

`capacity_year` is the natural key, and in an SCD2 dimension a natural key
repeats: a revised figure for 2025 would produce a second row carrying the same
year with a different validity period. A surrogate key is what identifies the
row. This is the actual reason surrogate keys exist, rather than convention.

## Point-in-time join

`fact_power_hour` joins the capacity dimension on a range, not on equality:

    time_utc >= valid_from AND time_utc < valid_to

Each hour is matched to the capacity version in force at that moment. Joining
on `is_current = true` instead would have applied 2026 capacity to 2025 hours
and understated every capacity factor in the first part of the series.

SCD2 without a point-in-time join does nothing useful. The two belong together.

Validation: the fact has 8,624 rows, the same as `gold_hourly`, so the range
join multiplied nothing, and zero rows failed to match a capacity version, so
the validity periods cover the whole series without gaps.

## Provenance of the capacity figures

Real values from the ENTSO-E Transparency Platform, documentType A68
(installed generation capacity aggregated), processType A33, psrType B19 (wind
onshore), domain 10YFI-1--------U. Fetched by `ingest/fetch_capacity.py`, which
is in the repository so the figures can be re-derived rather than trusted.

The values are typed into the notebook rather than landed through a volume,
because there are two of them and ENTSO-E is not reachable from Databricks Free
Edition. The source and query are recorded above so the numbers are verifiable.

In [0]:
from pyspark.sql import functions as F

SCHEMA = "workspace.energy_weather"

dim_date = (
    spark.table(f"{SCHEMA}.gold_hourly")
    # Calendar questions are asked in local time: "which day was it" means
    # the Finnish day, not the UTC day. The facts must key on the same basis.
    .select(F.to_date("time_local").alias("date"))
    .distinct()
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    # Spark numbers days 1=Sunday through 7=Saturday
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
    .withColumn(
        "season",
        F.when(F.month("date").isin(12, 1, 2), "winter")
        .when(F.month("date").isin(3, 4, 5), "spring")
        .when(F.month("date").isin(6, 7, 8), "summer")
        .otherwise("autumn"),
    )
    .select(
        "date_key", "date", "year", "month", "month_name", "day",
        "day_of_week", "day_name", "is_weekend", "season",
    )
)

dim_date.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA}.dim_date")

print("Rows:", dim_date.count())
dim_date.orderBy("date_key").show(3, truncate=False)

In [0]:
dim_area = (
    spark.table(f"{SCHEMA}.silver_weather")
    .select("area_id", "city")
    .distinct()
    # Region labels for the four observation points. These are descriptive,
    # not administrative: Finland has 19 official regions and this is a
    # four-point simplification chosen to span the country.
    .withColumn(
        "region",
        F.when(F.col("area_id") == "FI_S", "South")
        .when(F.col("area_id") == "FI_W", "West coast")
        .when(F.col("area_id") == "FI_E", "East")
        .otherwise("North"),
    )
    .withColumn(
        "wind_capacity_note",
        F.when(
            F.col("area_id") == "FI_W",
            "Most Finnish wind capacity is concentrated on the west coast",
        ).otherwise(None),
    )
    .select("area_id", "city", "region", "wind_capacity_note")
)

dim_area.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA}.dim_area")

dim_area.orderBy("area_id").show(truncate=False)

In [0]:
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.window import Window

SCHEMA = "workspace.energy_weather"

# Real figures from the ENTSO-E Transparency Platform:
#   documentType A68 (installed generation capacity aggregated)
#   processType A33, psrType B19 (wind onshore), in_Domain 10YFI-1--------U
# Fetched 2026-09-24 by ingest/fetch_capacity.py. Validity boundaries are the
# period boundaries reported by the source document itself.
capacity_rows = [
    (
        2025,
        8224.0,
        datetime(2024, 12, 31, 22, tzinfo=timezone.utc),
        datetime(2025, 12, 31, 22, tzinfo=timezone.utc),
    ),
    (
        2026,
        9330.0,
        datetime(2025, 12, 31, 22, tzinfo=timezone.utc),
        datetime(2026, 12, 31, 22, tzinfo=timezone.utc),
    ),
]

dim_wind_capacity = (
    spark.createDataFrame(
        capacity_rows,
        ["capacity_year", "installed_capacity_mw", "valid_from", "valid_to"],
    )
    # Surrogate key. The natural key alone cannot identify a row in an SCD2
    # dimension, because a revised figure for the same year would produce a
    # second row carrying that same natural key.
    .withColumn("capacity_key", F.row_number().over(Window.orderBy("valid_from")))
    .withColumn("is_current", F.col("valid_to") > F.current_timestamp())
    .select(
        "capacity_key",
        "capacity_year",
        "installed_capacity_mw",
        "valid_from",
        "valid_to",
        "is_current",
    )
)

dim_wind_capacity.write.format("delta").mode("overwrite").saveAsTable(
    f"{SCHEMA}.dim_wind_capacity"
)

dim_wind_capacity.orderBy("capacity_key").show(truncate=False)

In [0]:
capacity = spark.table(f"{SCHEMA}.dim_wind_capacity").select(
    "capacity_key", "installed_capacity_mw", "valid_from", "valid_to"
)

fact_power_hour = (
    spark.table(f"{SCHEMA}.gold_hourly")
    .select("time_utc", "time_local", "consumption_mw", "wind_mw", "price_eur_mwh")
    # Date key is derived from local time, matching how dim_date was built
    .withColumn("date_key", F.date_format("time_local", "yyyyMMdd").cast("int"))
    .withColumn("hour_of_day", F.hour("time_local"))
    # Point-in-time join: match each hour to the capacity version that was
    # in force at that moment, not to the current one
    .join(
        capacity,
        (F.col("time_utc") >= F.col("valid_from"))
        & (F.col("time_utc") < F.col("valid_to")),
        how="left",
    )
    # Capacity factor: how much of the installed capacity actually produced.
    # This is the reason the capacity dimension exists.
    .withColumn(
        "capacity_factor", F.col("wind_mw") / F.col("installed_capacity_mw")
    )
    .select(
        "time_utc",
        "time_local",
        "date_key",
        "hour_of_day",
        "capacity_key",
        "consumption_mw",
        "wind_mw",
        "price_eur_mwh",
        "capacity_factor",
    )
)

fact_power_hour.write.format("delta").mode("overwrite").saveAsTable(
    f"{SCHEMA}.fact_power_hour"
)

print("Rows:", fact_power_hour.count())
print("Unmatched capacity:", fact_power_hour.filter(F.col("capacity_key").isNull()).count())
fact_power_hour.orderBy("time_utc").show(3, truncate=False)

In [0]:
power_hours = spark.table(f"{SCHEMA}.fact_power_hour").select("time_utc")

fact_weather_hour = (
    spark.table(f"{SCHEMA}.silver_weather")
    # Keep only the hours the power fact also covers, so the two facts can be
    # analysed together through dim_date without partial days appearing
    .join(power_hours, on="time_utc", how="left_semi")
    .withColumn("date_key", F.date_format("time_local", "yyyyMMdd").cast("int"))
    .withColumn("hour_of_day", F.hour("time_local"))
    .select(
        "time_utc",
        "time_local",
        "date_key",
        "hour_of_day",
        "area_id",
        "temperature_c",
        "wind_speed_ms",
    )
)

fact_weather_hour.write.format("delta").mode("overwrite").saveAsTable(
    f"{SCHEMA}.fact_weather_hour"
)

print("Rows:", fact_weather_hour.count())
fact_weather_hour.orderBy("time_utc", "area_id").show(4, truncate=False)

# Report queries

Three questions, one per source combination, plus the synthesis that needs all
three at once. Figures below come from the queries that follow.

## Does temperature explain consumption

Yes, strongly and monotonically. Mean consumption by national average
temperature:

| Temperature | Mean consumption | Hours |
| --- | --- | --- |
| below -20 | 14,525 MW | 47 |
| -20 to -10 | 13,288 MW | 817 |
| -10 to 0 | 11,742 MW | 1,255 |
| 0 to +10 | 9,993 MW | 3,333 |
| +10 to +20 | 8,515 MW | 2,997 |
| above +20 | 8,793 MW | 175 |

A cold hour consumes roughly 70 percent more than a mild one.

**The warmest band appears to break the trend, and does not.** Consumption rises
again above +20. Holding time of day constant at 13:00 removes the reversal
(9,041 against 9,103 MW), because hours above +20 are almost all summer
afternoons while the +10 to +20 band contains spring and autumn nights. The
comparison was between times of day, not between temperatures.

This does not prove cooling load is zero. It shows the apparent effect was
mostly a confounder, and the remaining difference of 62 MW over 15 hours is
within noise.

**A note on the national average.** Summed per city, there are 492 hours below
-20 across the four points: Oulu 263, Kuopio 150, Helsinki 44, Vaasa 35. The
national average produces only 47. Averaging four points discards roughly 90
percent of locally extreme cold, because one mild city cancels another's
extreme. This is the measured version of why silver does not average weather.

## Which region's wind explains production

Vaasa, as the model assumed. Correlation between local wind speed and national
capacity factor:

| City | Region | Correlation |
| --- | --- | --- |
| Vaasa | West coast | 0.761 |
| Oulu | North | 0.726 |
| Kuopio | East | 0.634 |
| Helsinki | South | 0.523 |

The ranking reconstructs the geography of Finnish wind capacity from the data
alone: it is concentrated along the west coast from Vaasa northward, and sparse
inland and in the south. Nothing in the model was told where the turbines are.

All four are positive and substantial because weather systems span the country,
so every point carries some signal. The gradient is the finding.

Capacity factor is used rather than raw MW because installed capacity grew from
8,224 to 9,330 MW during the period. Raw production mixes the wind effect with
the capacity effect; the capacity factor separates them. This is what the SCD2
dimension is for.

Unlike the price relationships below, this one has a known physical mechanism:
wind turns the rotor. It is a measurement of a causal link, not only a
correlation.

## Cold and calm together

Mean day-ahead price by temperature and wind, in EUR/MWh with cents per kWh in
brackets:

| | Calm | Moderate | Windy |
| --- | --- | --- | --- |
| **Cold** (below 0 C) | **147.3** (14.73) | 89.9 (8.99) | 40.3 (4.03) |
| **Mild** (0 to 15 C) | 77.6 (7.76) | 37.0 (3.70) | 10.7 (1.07) |
| **Warm** (above 15 C) | 46.1 (4.61) | 19.2 (1.92) | **9.1** (0.91) |

Nine cells, monotonic in both directions, no exceptions. A cold calm hour costs
**sixteen times** a warm windy one. Both factors act and they compound.

Holding temperature at cold and varying only wind, the price falls from 147.3 to
40.3, a **73 percent reduction**, while consumption barely moves (12,204 against
12,612 MW). Demand is unchanged and supply increases, so the price collapses.
That isolates the supply-side effect.

Cold calm hours are **885, about 10 percent of the year**. This is not a rare
extreme but a recurring condition, which is why a power system is dimensioned for
its tightest hours rather than its average ones. The peak hour reached 61.4 cents
per kWh.

**The first version of this analysis used two temperature bands and hid a real
effect.** "Mild" covered everything above zero, which put a 5 C autumn evening
and a 25 C summer afternoon in the same bucket although their prices differ
almost twofold. That version reported a fourteenfold spread. Splitting warm out
revealed the full gradient and moved the headline to sixteen.

It also explains a confounder that the two-band version could not. Within the old
mild band, consumption appeared to rise with wind, from 8,831 to 10,221 MW. Wind
is supply, not demand, so that cannot be causal. Windy hours in Finland are
autumn and winter storms sitting at the cold end of a band wide enough to also
contain summer. The band was too wide; the relationship was not wrong.

## The same result without any bucketing

February and March 2026 are a natural experiment inside the data. Two winter
months with similar demand and opposite wind:

| | Consumption | Capacity factor | Price |
| --- | --- | --- | --- |
| February 2026 | 13,032 MW | 0.19 | **137.2 EUR/MWh** |
| March 2026 | 11,056 MW | 0.42 | **27.8 EUR/MWh** |

Demand differs by 18 percent, wind by 118 percent, price by 393 percent. The same
conclusion as the table above, reached without choosing a single threshold.

July is the cheapest month at 15.4 EUR/MWh **despite having the year's lowest
wind**, because demand is lowest too. Price is set by the ratio of demand to
supply, not by either alone. That is the most precise statement this dataset
supports, and it is a better answer than "wind makes power cheap".

## What these results do not establish

Correlation, not causation, except where a physical mechanism is known.

Time of day, day of week, industrial activity and interconnector flows with
neighbouring bidding zones all act at the same time and none of them are in the
model. Two of the findings above were shown to be confounded once one such
variable was controlled for, which is reason to assume others are too.

Prices are day-ahead spot, and exclude transmission, tax and margin.


In [0]:
%sql
WITH national_weather AS (
  SELECT
    time_utc,
    AVG(temperature_c) AS temp_avg_c
  FROM workspace.energy_weather.fact_weather_hour
  GROUP BY time_utc
)
SELECT
  CASE
    WHEN w.temp_avg_c < -20 THEN '1: below -20'
    WHEN w.temp_avg_c < -10 THEN '2: -20 to -10'
    WHEN w.temp_avg_c <   0 THEN '3: -10 to 0'
    WHEN w.temp_avg_c <  10 THEN '4: 0 to +10'
    WHEN w.temp_avg_c <  20 THEN '5: +10 to +20'
    ELSE                         '6: above +20'
  END                            AS temperature_band,
  ROUND(AVG(p.consumption_mw))   AS avg_consumption_mw,
  COUNT(*)                       AS hours
FROM workspace.energy_weather.fact_power_hour p
JOIN national_weather w
  ON p.time_utc = w.time_utc
-- Hold time of day constant, so the comparison is between temperatures
-- rather than between a summer afternoon and an autumn night
WHERE p.hour_of_day = 13
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT
  a.city,
  ROUND(MIN(w.temperature_c), 1)              AS coldest_c,
  COUNT_IF(w.temperature_c < -20)             AS hours_below_minus20
FROM workspace.energy_weather.fact_weather_hour w
JOIN workspace.energy_weather.dim_area a
  ON w.area_id = a.area_id
GROUP BY a.city
ORDER BY coldest_c

In [0]:
%sql
SELECT
  a.city,
  a.region,
  ROUND(CORR(w.wind_speed_ms, p.capacity_factor), 3) AS correlation,
  COUNT(*)                                           AS hours
FROM workspace.energy_weather.fact_weather_hour w
JOIN workspace.energy_weather.fact_power_hour p
  ON w.time_utc = p.time_utc
JOIN workspace.energy_weather.dim_area a
  ON w.area_id = a.area_id
GROUP BY a.city, a.region
ORDER BY correlation DESC

In [0]:
%sql
WITH national_weather AS (
  SELECT
    time_utc,
    AVG(temperature_c) AS temp_avg_c
  FROM workspace.energy_weather.fact_weather_hour
  GROUP BY time_utc
)
SELECT
  CASE
    WHEN w.temp_avg_c < 0 THEN 'A: cold (below 0)'
    ELSE                       'B: mild (0 or above)'
  END AS temperature,
  CASE
    WHEN p.capacity_factor < 0.15 THEN '1: calm (CF under 15%)'
    WHEN p.capacity_factor < 0.45 THEN '2: moderate'
    ELSE                               '3: windy (CF 45% or more)'
  END AS wind_conditions,
  COUNT(*)                        AS hours,
  ROUND(AVG(p.consumption_mw))    AS avg_consumption_mw,
  ROUND(AVG(p.price_eur_mwh), 2)  AS avg_price,
  ROUND(MAX(p.price_eur_mwh), 2)  AS max_price
FROM workspace.energy_weather.fact_power_hour p
JOIN national_weather w
  ON p.time_utc = w.time_utc
GROUP BY 1, 2
ORDER BY 1, 2